
# Module 2: QSAR classification model (step by step)


In this module, we will develop our first QSAR model, specifically a model for the acute toxicity of Earthworms. Because developing a QSAR model implies a workflow with several steps and this is your first QSAR model, this is going to be a long module where we will explore each step carefully. Hence, the workflow is divided in different lessons, and the corresponding Python practice is divided into different Jupyter Notebook files. As a reminder, in this course the workflow for developing a QSAR model is dividided in the following parts:

 - Part 1: Data retrieval and curation
   
 - Part 2: Calculation of molecular descriptors

 - Part 3: Train/Test splitting and standardisation
    
 - Part 4: Feature selection

 - Part 5: Model development and optimisation

 - Part 6: Prediction and applicability domain


# Part 1: Data retrieval and curation


In this lesson, we will learn how to pre-process the database to obtain a suitable set of molecules to build a model.

First, we will read the database, select which data we want to use in our analysis, and, if necessary, transform the endpoint values to have a consistent dataset ([Section 1](#scrollTo=71bad2f6)).

Afterward, we will perform a data curation procedure to remove all the molecules that are described by an incorrect SMILES, that are not adequate for our model, or that are duplicated ([Section 2](#scrollTo=6a6178e0)).

## Preliminary: Preparation of the environment
Before proceeding to develop the QSAR model, we need to set up the notebook to have access to the necessary packages and modules. Please execute the following cells to prepare the environment.

First, we need to install some packages that are not available in Google Colab by default. Colab allows the installation of packages directly through pip. We will install the RDKit package, as we will use it to read the molecules and assess their validity.


In [28]:
#@title Install RDKit
!pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.8/31.8 MB 21.6 MB/s eta 0:00:00


Also, we are going to read the data from a file and later save and read different intermediate files. Hence, it is necessary to have access to the different files needed for the lessons.

You can upload local files using the files panel on the right (the fourth symbol in the left menu with a folder shape), but also using the code below (which will also make the file appear in the panel). Try the code below if you find errors using the upload menu, as it sometimes works to solve the issues.


In [21]:
#@title Mount Google Drive
from google.colab import drive
import os

# Mount Google Drive to access your files
drive.mount('/content/drive')

Mounted at /content/drive


In [25]:
#@title File I/O setup
import os

## Modify and uncomment the appropiate lines below to mark the right path to the folder where all the files are going to be stored

# We will use the local Colab folder since the file is located there:
PATH = "/content/"

# if you use GoogleDrive, decomment next lines (You will need to give permission to Colab to access your Drive account):
#PATH = "/content/drive/MyDrive/PQSAR_Python_Course/"


## Upload the file to your folder 'Earthworm_acute_toxicity_raw_data.csv'and check if it exists:
os.path.isfile(PATH+'Earthworm_acute_toxicity_raw_data.csv')

True

To start, we are going to execute a cell in the notebook to import the basic modules that we will use in this lesson. This can be done at the beginning if you already know what you are going to use, or just before using a new module.

In [29]:
#@title Import required packages
import pandas as pd
from rdkit import Chem

## Section 1. Reading a data set of molecules and preparing a dataset for modelling

The first step in creating a QSAR model is to obtain and select the data to be used. Compiling or finding a good database for your model is very important.

The exact way of accessing the data will depend on the source of the data and the format of our database. We have to consider questions such as:
  - Can we directly download it, or do we need and API?
  - Do we have a single file, or do we need to combine multiple files?
  - What is the file extension?
  - How are the molecules represented?
  - Do we have extra data fields with useful information?


Data can be find in multiple formats, but it is very common to be able to download compound databases as one or more tables. A very common file format for tables is the comma-separated values (**CSV**) format (check this link https://docs.fileformat.com/spreadsheet/csv/ for more details). This is the one we will use throughout this course.

Check how we use the cell below to read the file **Earthworm_acute_toxicity_raw_data.csv** from the folder stored in `PATH`. In this case, we will open the database as Pandas DataFrame using the `read_csv` method of Pandas. Note that we use the `PATH` variable in front of the file name to access the folder determined earlier and that we request to use the semicolon (";") as the separator.

In [30]:
#@title Import data from csv

df_classification = pd.read_csv(PATH+'Earthworm_acute_toxicity_raw_data.csv', sep=';')

df_classification.head() # Show the first rows to check it is the adequate file

,SMILES,value,Experimental guidelines
0,COc1nc(C)nc(NC(=O)NS(=O)(=O)c2ccccc2OCCCl)n1,Non-toxic,"OECD Guidelines 207: Earthworm, Acute Toxicity..."
1,CCO/N=C(\CC)C1=C(O)CC(c2c(C)cc(C)cc2C)CC1=O,Positive,"OECD Guidelines 207: Earthworm, Acute Toxicity..."
2,CC1(C(=O)Nc2ccc(O)c(Cl)c2Cl)CCCCC1,Non-toxic,"OECD Guidelines 207: Earthworm, Acute Toxicity..."
3,O=C(NC(=O)c1c(F)cccc1F)Nc1cc(Cl)c(OC(F)(F)C(F)...,non-toxic,"OECD Guidelines 207: Earthworm, Acute Toxicity..."
4,O=[N+]([O-])c1cc(C(F)(F)F)c(Cl)c([N+](=O)[O-])...,negative,"OECD Guidelines 207: Earthworm, Acute Toxicity..."


In [31]:
#@title Check the name of the columns

# Get and print the column names
print(df_classification.columns)

Index(['SMILES', 'value', 'Experimental guidelines'], dtype='object')


You should have seen that there is a field in the database regarding experimental protocol. It is important to check this kind of data and to evaluate which is better: to restrict to one single method (this reduces the noise but decreases the amount of data) or to combine several.

The first step is to evaluate the distribution of the values, the fastest and easier way is to use the `value_counts` method (https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html).

In [34]:
#@title Check experimental protocol values

# Calculate and print the frequency of each experimental protocol
df_classification['Experimental guidelines'].value_counts()

,count
Experimental guidelines,
"OECD Guidelines 207: Earthworm, Acute Toxicity Tests",174
850.3100 - Earthworm Subchronic Toxicity Test,28


In this case, the database includes different options, but there is a significant majority of values for a single protocol. Furthermore, this is an OECD standard protocol, which ensures these values are obtained following a standardized method, and their reliability is established. Thus, in this case, we will restrict our model to the data obtained by this method.

Create a new dataframe `df_classification_filtered` which has only the records corresponding to this method.

In [35]:
#@title Filter the database

# Filter the database to keep only the OECD Guidelines 207 protocol
df_classification_filtered = df_classification[df_classification['Experimental guidelines'] == 'OECD Guidelines 207: Earthworm, Acute Toxicity Tests']

#Show at the end the number of rows, to check that the filter was successful
df_classification_filtered.shape[0]

174

Now, we are going to check the values of the endpoints. We need to check three different aspects:
  1. How are the values labeled? Do we already have a classification as required for this model? Are the categories labeled in a consistent way?
  1. Do we have any empty, incorrect, or unclear values that we need to remove?
  1. Do we have enough positive and negative values? For modeling the ideal scenario is to have a balanced dataset.

Visualize the data distribution for the endpoint value (you can use the same approach we used for the experimental protocol).

In [36]:
#@title Check labels for endpoint

# Visualize the distribution of the toxic/non-toxic labels
df_classification_filtered['value'].value_counts()

,count
value,
negative,31
Negative,31
toxic,29
Non-toxic,24
Positive,23
positive,15
non-toxic,11
Toxic,10


In this case, we have a significant variation in the labels. Thus, you need to identify them in a consistent way.

It is recommended to use numeric values (this is not a requirement for all algorithms, but it could be for some, so it is better to use numbers consistently). We will use 1 for "toxic" or "positive" values and 0 for "non-toxic" or negative values.

Convert the values to 1/0 and store them in a column named 'y' (which is a typical name for the endpoint in machine learning).

There are several ways to do this. You can prepare a dictionary with the equivalence of the different values mapped to 1 and 0 and then use the `map` method to convert the '**value**' column (https://pandas.pydata.org/docs/dev/reference/api/pandas.DataFrame.map.html). Alternatively, you can change the labels to numbers one by one using the `replace` method (https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.replace.html).

In [41]:
#@title Harmonize endpoint labels

# Dictionary mapping all text variations to binary labels (1 for toxic/positive, 0 for non-toxic/negative)
label_map = {
    'toxic': 1, 'Toxic': 1,
    'positive': 1, 'Positive': 1,
    'non-toxic': 0, 'Non-toxic': 0,
    'negative': 0, 'Negative': 0
}

# Create the new target column 'y'
df_classification_filtered['y'] = df_classification_filtered['value'].map(label_map)

# Print your DataFrame to check your progress
df_classification_filtered

,SMILES,value,Experimental guidelines,y
0,COc1nc(C)nc(NC(=O)NS(=O)(=O)c2ccccc2OCCCl)n1,Non-toxic,"OECD Guidelines 207: Earthworm, Acute Toxicity...",0
1,CCO/N=C(\CC)C1=C(O)CC(c2c(C)cc(C)cc2C)CC1=O,Positive,"OECD Guidelines 207: Earthworm, Acute Toxicity...",1
2,CC1(C(=O)Nc2ccc(O)c(Cl)c2Cl)CCCCC1,Non-toxic,"OECD Guidelines 207: Earthworm, Acute Toxicity...",0
3,O=C(NC(=O)c1c(F)cccc1F)Nc1cc(Cl)c(OC(F)(F)C(F)...,non-toxic,"OECD Guidelines 207: Earthworm, Acute Toxicity...",0
4,O=[N+]([O-])c1cc(C(F)(F)F)c(Cl)c([N+](=O)[O-])...,negative,"OECD Guidelines 207: Earthworm, Acute Toxicity...",0
...,...,...,...,...
197,O=C(O)/C=C/C(=O)O.O=C(c1ccc(F)c(F)c1Nc1ccc(I)c...,positive,"OECD Guidelines 207: Earthworm, Acute Toxicity...",1
198,O=C(c1ccc(F)c(F)c1Nc1ccc(I)cc1F)N1CC(O)([C@@H]...,positive,"OECD Guidelines 207: Earthworm, Acute Toxicity...",1
199,Cc1c(F)cc(C(=O)NC2CC2)cc1-c1ccc(C(=O)NCC(C)(C)...,positive,"OECD Guidelines 207: Earthworm, Acute Toxicity...",1
200,CClCCC=OF,negative,"OECD Guidelines 207: Earthworm, Acute Toxicity...",0


Once the dataset is filtered and ready, we will select only the columns of interest and save it as a .csv file.

In [42]:
#@title Save the files

df_classification_final = df_classification_filtered[['SMILES', 'y']]

df_classification_final.to_csv(PATH+'Earthworm_acute_toxicity-preprocessed.csv', sep=';', index=False)

## SECTION 2. Cleaning the molecules in a data set

Now that we know how to import the data, let's start curating the dataset to filter those values that are not correct or practical for their use in the development of a model.

We will be working with the Earthworm acute toxicity filtered dataset that we created and saved in the CSV file "**Earthworm_acute_toxicity-preprocessed.csv**" in the previous step. Let's retrieve the dataset again from the CSV file.

Note that this step is relatively unnecessary because we already have the DataFrame in memory (it was just saved). However, it is recommended that you read it again, in case you need to resume your work at another moment or to be able to repeat **Section 2**, ensuring that the initial data is consistent across runs.

In [43]:
#@title Retrieve the selected data

# Read the preprocessed CSV file using pandas
df_classification = pd.read_csv(PATH + 'Earthworm_acute_toxicity-preprocessed.csv', sep=';')

# Visualize to check
df_classification

,SMILES,y
0,COc1nc(C)nc(NC(=O)NS(=O)(=O)c2ccccc2OCCCl)n1,0
1,CCO/N=C(\CC)C1=C(O)CC(c2c(C)cc(C)cc2C)CC1=O,1
2,CC1(C(=O)Nc2ccc(O)c(Cl)c2Cl)CCCCC1,0
3,O=C(NC(=O)c1c(F)cccc1F)Nc1cc(Cl)c(OC(F)(F)C(F)...,0
4,O=[N+]([O-])c1cc(C(F)(F)F)c(Cl)c([N+](=O)[O-])...,0
...,...,...
169,O=C(O)/C=C/C(=O)O.O=C(c1ccc(F)c(F)c1Nc1ccc(I)c...,1
170,O=C(c1ccc(F)c(F)c1Nc1ccc(I)cc1F)N1CC(O)([C@@H]...,1
171,Cc1c(F)cc(C(=O)NC2CC2)cc1-c1ccc(C(=O)NCC(C)(C)...,1
172,CClCCC=OF,0


Now, we are going to apply this technique to the DataFrame for our model.
Iterate through the elements in the SMILES column and create a list called `new_smiles` with the sanitized SMILES, and an empty value (None) for the incorrect ones.

After that, use this list to create a new column and filter this column using, for example, the `notna` method of pandas (https://pandas.pydata.org/docs/reference/api/pandas.notna.html).

You resulting DataFrame should have **166** molecules.

In [47]:
#@title Sanitization in the database

#Create a list to include the sanitized SMILES
new_smiles = []

#Iterate on the list of SMILES
for smi in df_classification['SMILES']:
    try:
        #Sanitize if possible by creating a molecule and generating canonical SMILES
        mol = Chem.MolFromSmiles(smi)
        if mol is not None:
            sanitized_smi = Chem.MolToSmiles(mol)
        else:
            sanitized_smi = None
    except:
        #Use None if not (None without "" because it is an empty value not a text)
        sanitized_smi = None
    #Append the value to the new_smiles list
    new_smiles.append(sanitized_smi)

#Include the new_smiles list as a new 'san_SMILES' column
df_classification['san_SMILES'] = new_smiles

# Filter out invalid SMILES (keep only rows where san_SMILES is not null)
df_classification = df_classification[df_classification['san_SMILES'].notna()]

df_classification.shape

[12:35:35] WARNING: not removing hydrogen atom without neighbors
[12:35:35] WARNING: not removing hydrogen atom without neighbors
[12:35:35] WARNING: not removing hydrogen atom without neighbors
[12:35:35] WARNING: not removing hydrogen atom without neighbors
[12:35:35] Explicit valence for atom # 4 F, 2, is greater than permitted
[12:35:35] Explicit valence for atom # 1 Cl, 2, is greater than permitted
[12:35:35] Explicit valence for atom # 3 O, 4, is greater than permitted
[12:35:35] WARNING: not removing hydrogen atom without neighbors
[12:35:35] WARNING: not removing hydrogen atom without neighbors
[12:35:35] WARNING: not removing hydrogen atom without neighbors
[12:35:35] non-ring atom 0 marked aromatic
[12:35:35] non-ring atom 0 marked aromatic
[12:35:35] WARNING: not removing hydrogen atom without neighbors
[12:35:35] WARNING: not removing hydrogen atom without neighbors
[12:35:35] SMILES Parse Error: syntax error while parsing: CO!OO=C
[12:35:35] SMILES Parse Error: check for m

(166, 3)

### 2.2 Eliminate salts from the database

We need to eliminate salts from our molecules, as we train our models on single species. The first step is to simplify the salts that include very common counterions, by removing the counterions and keeping the main ion.

Check RDKit's documentation to see how to do this (https://www.rdkit.org/docs/source/rdkit.Chem.html).

In [48]:
#@title Example of how salts are removed
# Import the salt remover function from rdkit
from rdkit.Chem.SaltRemover import SaltRemover

# list of SMILES to analise
list_of_SMILES = ['N#Cc1c(Cl)cccc1Cl.[Na+]',
'CC(C)(C)C(O)C(Oc1ccc(-c2ccccc2)cc1)n1cncn1.[Cl-]',
'CC(C)N(C(=O)SCC(Cl)=C(Cl)Cl)C(C)C.[K+]',
'CC(C)Oc1cc(-n2nc(C(C)(C)C)oc2=O)c(Cl)cc1Cl.[OH-]',
'Clc1ccc(C2(Cn3cncn3)CC(Br)CO2)c(Cl)c1.[NH3+]',
'C=CCOC(Cn1ccnc1)c1ccc(Cl)cc1Cl.[I-]',
'CCOc1nc(C(Cl)(Cl)Cl)ns1.[F-]']

# Create a loop through the list of SMILES
for smi in list_of_SMILES:
    # Create a mol object from the SMILES
    mol = Chem.MolFromSmiles(smi)
    # Initialize the salt remover object
    remover = SaltRemover(defnData='[Na,Cl,K,O,OH,Fe,F,H,Al,Mg,Co,Ti,NH4,Mn,Si,Ca,Au,I,Hg,Mo,Zn,Br,Ag,Sr,Cu,Bi,S,Li,NH3,He,Y,Ar,Ba,La]')
    # Apply the remover on the SMILES using the StripMol method
    res = remover.StripMol(mol)
    # Create the new SMILES from the molecule without salt (res from previous step)
    new_smi = Chem.MolToSmiles(res)
    # Print the new SMILES to check the result
    print(new_smi)

N#Cc1c(Cl)cccc1Cl
CC(C)(C)C(O)C(Oc1ccc(-c2ccccc2)cc1)n1cncn1
CC(C)N(C(=O)SCC(Cl)=C(Cl)Cl)C(C)C
CC(C)Oc1cc(-n2nc(C(C)(C)C)oc2=O)c(Cl)cc1Cl
Clc1ccc(C2(Cn3cncn3)CC(Br)CO2)c(Cl)c1
C=CCOC(Cn1ccnc1)c1ccc(Cl)cc1Cl
CCOc1nc(C(Cl)(Cl)Cl)ns1


Apply what we have learned until now to remove the salts from our `df_classification` dataframe.

To facilitate the comparison of the results, please save the SMILES without salts in a new column called '**NO SALTS**'.

In [49]:
#@title Apply salt remover to the dataset
from rdkit.Chem.SaltRemover import SaltRemover

smi_no_salts = []
for smi in df_classification['SMILES']:
    mol = Chem.MolFromSmiles(smi)
    remover = SaltRemover(defnData='[Na,Cl,K,O,OH,Fe,F,H,Al,Mg,Co,Ti,NH4,Mn,Si,Ca,Au,I,Hg,Mo,Zn,Br,Ag,Sr,Cu,Bi,S,Li,NH3,He,Y,Ar,Ba,La]')
    res = remover.StripMol(mol)
    new_smi = Chem.MolToSmiles(res)
    smi_no_salts.append(new_smi)

df_classification['NO SALTS'] = smi_no_salts

[12:36:11] WARNING: not removing hydrogen atom without neighbors
[12:36:11] WARNING: not removing hydrogen atom without neighbors
[12:36:11] WARNING: not removing hydrogen atom without neighbors
[12:36:11] WARNING: not removing hydrogen atom without neighbors
[12:36:11] WARNING: not removing hydrogen atom without neighbors
[12:36:11] WARNING: not removing hydrogen atom without neighbors
[12:36:11] WARNING: not removing hydrogen atom without neighbors
[12:36:12] WARNING: not removing hydrogen atom without neighbors
[12:36:12] WARNING: not removing hydrogen atom without neighbors
/tmp/ipykernel_1896/2856302435.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_classification['NO SALTS'] = smi_no_salts


Before we continue, you should have noticed several warnings related to hydrogen atoms without neighbors. This is because the salt remover doesn't remove isolated protons. Thus, it would be good practice to remove them using a different approach (such as detecting the **"[H+]"** fragment in the SMILES). For simplicity, we will not do this in this course.  

Note that we call this part **"salt removal,"** but this does not mean that we remove the salts from our dataset—instead, we simplify their SMILES so we keep only the main ion. This contrasts with other steps later, such as removing inorganics and mixtures, which actually remove the data from the database.  

Nevertheless, it could happen that a substance needs to be eliminated in this process if it is a salt formed only by the common ions removed. Therefore, we will remove from the dataset all the cases where the "**NO SALTS**" column became empty.

Did this remove any substance? (You can check by printing the length of the database before and after.)

In [56]:
#@title Remove empty values after salt removal
df_classification = df_classification[df_classification['NO SALTS']!= '']
df_classification

,SMILES,y,san_SMILES,NO SALTS
0,COc1nc(C)nc(NC(=O)NS(=O)(=O)c2ccccc2OCCCl)n1,0,COc1nc(C)nc(NC(=O)NS(=O)(=O)c2ccccc2OCCCl)n1,COc1nc(C)nc(NC(=O)NS(=O)(=O)c2ccccc2OCCCl)n1
1,CCO/N=C(\CC)C1=C(O)CC(c2c(C)cc(C)cc2C)CC1=O,1,CCO/N=C(\CC)C1=C(O)CC(c2c(C)cc(C)cc2C)CC1=O,CCO/N=C(\CC)C1=C(O)CC(c2c(C)cc(C)cc2C)CC1=O
2,CC1(C(=O)Nc2ccc(O)c(Cl)c2Cl)CCCCC1,0,CC1(C(=O)Nc2ccc(O)c(Cl)c2Cl)CCCCC1,CC1(C(=O)Nc2ccc(O)c(Cl)c2Cl)CCCCC1
3,O=C(NC(=O)c1c(F)cccc1F)Nc1cc(Cl)c(OC(F)(F)C(F)...,0,O=C(NC(=O)c1c(F)cccc1F)Nc1cc(Cl)c(OC(F)(F)C(F)...,O=C(NC(=O)c1c(F)cccc1F)Nc1cc(Cl)c(OC(F)(F)C(F)...
4,O=[N+]([O-])c1cc(C(F)(F)F)c(Cl)c([N+](=O)[O-])...,0,O=[N+]([O-])c1cc(C(F)(F)F)c(Cl)c([N+](=O)[O-])...,O=[N+]([O-])c1cc(C(F)(F)F)c(Cl)c([N+](=O)[O-])...
...,...,...,...,...
159,CCC(=O)O[C@]1(C(=O)CCl)[C@@H](C)C[C@H]2[C@@H]3...,1,CCC(=O)O[C@]1(C(=O)CCl)[C@@H](C)C[C@H]2[C@@H]3...,CCC(=O)O[C@]1(C(=O)CCl)[C@@H](C)C[C@H]2[C@@H]3...
160,Cn1cnc2c(F)c(Nc3ccc(Br)cc3Cl)c(C(=O)NOCCO)cc21,1,Cn1cnc2c(F)c(Nc3ccc(Br)cc3Cl)c(C(=O)NOCCO)cc21,Cn1cnc2c(F)c(Nc3ccc(Br)cc3Cl)c(C(=O)NOCCO)cc21
161,O=C(O)/C=C/C(=O)O.O=C(c1ccc(F)c(F)c1Nc1ccc(I)c...,1,O=C(O)/C=C/C(=O)O.O=C(c1ccc(F)c(F)c1Nc1ccc(I)c...,O=C(O)/C=C/C(=O)O.O=C(c1ccc(F)c(F)c1Nc1ccc(I)c...
162,O=C(c1ccc(F)c(F)c1Nc1ccc(I)cc1F)N1CC(O)([C@@H]...,1,O=C(c1ccc(F)c(F)c1Nc1ccc(I)cc1F)N1CC(O)([C@@H]...,O=C(c1ccc(F)c(F)c1Nc1ccc(I)cc1F)N1CC(O)([C@@H]...


### 2.3 Eliminate inorganic and metalorganic molecules

The next step is to eliminate inorganic and metal-organic substances. To do this, we will define a list of allowed atoms and remove any substance that includes other atoms. You will determine which elements are present in each substance and iterate through them to compare them with the list. If any element is not on the list, we will remove the substance.

In this course, we will allow **H, N, C, O, P, S, Si, F, Cl, Br,** and **I**.

Explore this in the example below (two substances should stay and two should disappear).

In [51]:
#@title Example of inorganics removal

# List of SMILES to analise
list_of_SMILES = ['Cl[Fe](Cl)Cl',
'CC(C)(C)C(O)C(Oc1ccc(-c2ccccc2)cc1)n1cncn1',
'[N+](=O)([O-])[O-].[N+](=O)([O-])[O-].[Cu+2]',
'O=S(=O)(O)O']

# List of allowed atoms
allowed_atoms = ['H', 'N', 'C', 'O', 'P', 'S', 'Si', 'P', 'F', 'Cl', 'Br', 'I']

# Create a loop through the list of SMILES
for smi in list_of_SMILES[:]:  # Iterate on a copy of the list when modifying it
    # Create a mol object from the SMILES
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        list_of_SMILES.remove(smi)
        continue
    # Create a loop through the atoms of the molecule using GetAtoms()
    for atom in mol.GetAtoms():
        # Get the atom symbol using GetSymbol()
        atom_symbol = atom.GetSymbol()
        # Check if the atom is in the list of allowed atoms
        if atom_symbol not in allowed_atoms:
           # if is not an allowed atom remove the molecule
           list_of_SMILES.remove(smi)
           break

# Check the remaining SMILES
print(list_of_SMILES)

['CC(C)(C)C(O)C(Oc1ccc(-c2ccccc2)cc1)n1cncn1', 'O=S(=O)(O)O']


Once you think your code works properly, use it on the database.

Iterate through the elements in the "**NO SALTS**" column. To remove the data rows with non-allowed atoms, a practical way is to first create a list, for example, called "**non_allowed_indx**", which will contain the indices of the rows with SMILES that contain non-allowed atoms.

After that, you can use this list to drop these indices from the DataFrame using the `drop` method. This will not change the indices, so if you want to have a nicely ordered DataFrame with the index corresponding to the row, it is recommended to reset your index.

Alternatively, you can use an approach similar to the previous one and modify the SMILES in the "**NO SALTS**" column to an empty value.

In [52]:
#@title Inorganic removal in the dataset
allowed_atoms = ['H', 'N', 'C', 'O', 'P', 'S', 'Si', 'F', 'Cl', 'Br', 'I']

non_allowed_indx = []
#We reset the index to be sure that the index corresponds to the actual position of the SMILES in the list
df_classification.reset_index(drop=True, inplace=True)

#Use enumerate to obtain the index in additon of the SMILES
for i, smi in enumerate(df_classification['NO SALTS']):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        non_allowed_indx.append(i)
        continue
    for atom in mol.GetAtoms():
        atom_symbol = atom.GetSymbol()
        if atom_symbol not in allowed_atoms:
           non_allowed_indx.append(i)  # Append the index
           break

df_classification.drop(non_allowed_indx,inplace=True)
df_classification.reset_index(drop=True, inplace=True)

df_classification

[12:36:55] WARNING: not removing hydrogen atom without neighbors
[12:36:55] WARNING: not removing hydrogen atom without neighbors
[12:36:55] WARNING: not removing hydrogen atom without neighbors
[12:36:55] WARNING: not removing hydrogen atom without neighbors
[12:36:55] WARNING: not removing hydrogen atom without neighbors
[12:36:55] WARNING: not removing hydrogen atom without neighbors
[12:36:55] WARNING: not removing hydrogen atom without neighbors
[12:36:55] WARNING: not removing hydrogen atom without neighbors
[12:36:55] WARNING: not removing hydrogen atom without neighbors
/tmp/ipykernel_1896/282336055.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_classification.drop(non_allowed_indx,inplace=True)


,SMILES,y,san_SMILES,NO SALTS
0,COc1nc(C)nc(NC(=O)NS(=O)(=O)c2ccccc2OCCCl)n1,0,COc1nc(C)nc(NC(=O)NS(=O)(=O)c2ccccc2OCCCl)n1,COc1nc(C)nc(NC(=O)NS(=O)(=O)c2ccccc2OCCCl)n1
1,CCO/N=C(\CC)C1=C(O)CC(c2c(C)cc(C)cc2C)CC1=O,1,CCO/N=C(\CC)C1=C(O)CC(c2c(C)cc(C)cc2C)CC1=O,CCO/N=C(\CC)C1=C(O)CC(c2c(C)cc(C)cc2C)CC1=O
2,CC1(C(=O)Nc2ccc(O)c(Cl)c2Cl)CCCCC1,0,CC1(C(=O)Nc2ccc(O)c(Cl)c2Cl)CCCCC1,CC1(C(=O)Nc2ccc(O)c(Cl)c2Cl)CCCCC1
3,O=C(NC(=O)c1c(F)cccc1F)Nc1cc(Cl)c(OC(F)(F)C(F)...,0,O=C(NC(=O)c1c(F)cccc1F)Nc1cc(Cl)c(OC(F)(F)C(F)...,O=C(NC(=O)c1c(F)cccc1F)Nc1cc(Cl)c(OC(F)(F)C(F)...
4,O=[N+]([O-])c1cc(C(F)(F)F)c(Cl)c([N+](=O)[O-])...,0,O=[N+]([O-])c1cc(C(F)(F)F)c(Cl)c([N+](=O)[O-])...,O=[N+]([O-])c1cc(C(F)(F)F)c(Cl)c([N+](=O)[O-])...
...,...,...,...,...
159,CCC(=O)O[C@]1(C(=O)CCl)[C@@H](C)C[C@H]2[C@@H]3...,1,CCC(=O)O[C@]1(C(=O)CCl)[C@@H](C)C[C@H]2[C@@H]3...,CCC(=O)O[C@]1(C(=O)CCl)[C@@H](C)C[C@H]2[C@@H]3...
160,Cn1cnc2c(F)c(Nc3ccc(Br)cc3Cl)c(C(=O)NOCCO)cc21,1,Cn1cnc2c(F)c(Nc3ccc(Br)cc3Cl)c(C(=O)NOCCO)cc21,Cn1cnc2c(F)c(Nc3ccc(Br)cc3Cl)c(C(=O)NOCCO)cc21
161,O=C(O)/C=C/C(=O)O.O=C(c1ccc(F)c(F)c1Nc1ccc(I)c...,1,O=C(O)/C=C/C(=O)O.O=C(c1ccc(F)c(F)c1Nc1ccc(I)c...,O=C(O)/C=C/C(=O)O.O=C(c1ccc(F)c(F)c1Nc1ccc(I)c...
162,O=C(c1ccc(F)c(F)c1Nc1ccc(I)cc1F)N1CC(O)([C@@H]...,1,O=C(c1ccc(F)c(F)c1Nc1ccc(I)cc1F)N1CC(O)([C@@H]...,O=C(c1ccc(F)c(F)c1Nc1ccc(I)cc1F)N1CC(O)([C@@H]...


### 2.4 Remove mixtures

**Mixed molecules** are coded in **SMILES** by a dot **separating** the two molecules, for example, **"CCO.O"** (**ethanol and water**). To calculate descriptors, it is important that we have **SMILES** coding only **pure** molecules. **So,** we will remove **mixtures** in this step (**this also includes** the removal of any salt that **remained** in the salt removal step if it **contains** more than one component).  

A good approach is to **manually** explore **mixtures** to detect the main component and keep it. **For example,** it is often possible to remove common excipients such as **water ('O'), ethanol ('CCO'),** and **DMSO ('CS(=O)C').** However, to simplify the process in this course, we will remove **all** mixtures.  

**Please remove any mixtures from the** `df_classification` **DataFrame**. A simple way is to use the **"."** character in the **SMILES** as a signal for mixtures.  


In [ ]:
#@title Mixtures removal in the dataset
mixtures_indx = []

for i, smi in enumerate(df_classification['NO SALTS']):
    if '.' in smi:
        mixtures_indx.append(i)

df_classification.drop(mixtures_indx, inplace=True)
df_classification.reset_index(drop=True, inplace=True)

### 2.5 Remove duplicated molecules

In order to remove the duplicates there is a Pandas built-in function that performs this work.
In order to check more information about how this function works you can visit https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop_duplicates.html

The main parameters for this function are:
* `subset`: column (or columns) to consider for identifying duplicates.
* `keep`: Determines if you want to keep the `first` or the `last` duplicate (or no duplicate if `False`).

**Note**: If we decide to keep one of the duplicate molecules, we must first check whether they have the same output variable. Keeping duplicates with conflicting labels could introduce bias and inconsistencies in our dataset. To avoid this issue and simplify the process, we will remove all duplicate molecules in this case.

In [53]:
#@title Removal of duplicates in the dataset
df_classification.drop_duplicates(subset=['NO SALTS'], keep=False, inplace=True)

/tmp/ipykernel_1896/1633522937.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_classification.drop_duplicates(subset=['NO SALTS'], keep=False, inplace=True)


###**Q3.** Let's do a final check. How many molecules do you have in your curated dataset?


In [54]:
#@title Check remaining molecules
df_classification.shape

(142, 4)

### 2.6 Select and rename the columns to create the final dataset

With the **SMILES** **sanitized** and the list of **compounds** cleaned of salts, duplicates, etc., it's time to create a final dataset with the **final** list of **SMILES** and their response values.  

It is important to save the **sanitized** database as **a** file to be used in the following steps of the process and to have a copy to document your model.

In [55]:
#@title Save the curated database
# Create a df_classification_final dataframe with the columns 'NO SALTS' and 'y'
df_classification_final = df_classification[['NO SALTS', 'y']]

# Rename the columns as 'SMILES' and 'y'
df_classification_final.columns = ['SMILES', 'y']

# Save the resulting dataframe into a csv file called Earthworm_acute_toxicity.csv
df_classification_final.to_csv(PATH+'Earthworm_acute_toxicity.csv', sep=';', index=False)